# Extract clips from .avi

Fill in the config cell, then run all cells.

Each clip is a dict with:
- `name`  — output filename stem (no extension)
- `start` — start time in **seconds** (float) or **frames** (int)
- `stop`  — stop time in **seconds** (float) or **frames** (int)
- `unit`  — `'sec'` or `'frames'` (default: `'sec'`)

Set `mark_flies = True` to overlay fly positions, orientation arrows, and ID labels on each frame.  
Tracking data is loaded automatically from `rootdir/processed_mats/<acq>.parquet`.

Clips are saved to `rootdir/figures/<acq>/clips/`.

In [2]:
import cv2
import os
import math
import numpy as np
import pandas as pd

In [33]:
# ── CONFIG ──────────────────────────────────────────────────────────────────

rootdir = "/Users/juliechen/Library/CloudStorage/Dropbox-Dropbox@RU/Julie Chen/Ruta lab rotation 2026/fb_20mm/"
acq     = "20260511-1000_MMF-triad3_Dyak__4do_gh"

clips = [
    {'name': 'courting_over_deF', 'start': 76000, 'stop': 77000, 'unit': 'frames'},
    {'name': 'guarding_circling', 'start': 98300, 'stop': 98800, 'unit': 'frames'}
    #{'name': 'guarding_circling', 'start': 21100, 'stop': 22000, 'unit': 'frames'},
]

mark_flies = False   # set False to skip fly overlay

# fly id → BGR color  (dodgerblue, tomato, limegreen, yellow)
id_colors = {
    0: (255, 144, 30),
    1: (71,  99,  255),
    2: (50,  205, 50),
    3: (0,   215, 255),
}

# ── derived paths (no need to edit below) ───────────────────────────────────
avi_path  = os.path.join(rootdir, 'raw_videos', acq, f'{acq}.avi')
trk_path  = os.path.join(rootdir, 'processed_mats', f'{acq}.parquet')
save_dir  = os.path.join(rootdir, 'figures', acq, 'clips')
os.makedirs(save_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────

In [34]:
# Preview video properties
cap = cv2.VideoCapture(avi_path)
fps        = cap.get(cv2.CAP_PROP_FPS)
n_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_w    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

duration_s = n_frames / fps
print(f"Video: {os.path.basename(avi_path)}")
print(f"  {frame_w}x{frame_h}  |  {fps:.1f} fps  |  {n_frames} frames  |  {duration_s:.1f} s ({duration_s/60:.1f} min)")

Video: 20260511-1000_MMF-triad3_Dyak__4do_gh.avi
  1088x1100  |  60.0 fps  |  108000 frames  |  1800.0 s (30.0 min)


In [35]:
# Load tracking data and build per-frame lookup (skipped when mark_flies=False)
frame_lookup = {}

if mark_flies:
    assert os.path.exists(trk_path), f"Parquet not found: {trk_path}"
    trk = pd.read_parquet(trk_path)
    print(f"Loaded: {os.path.basename(trk_path)}  ({len(trk)} rows, "
          f"frames {trk['frame'].min()}–{trk['frame'].max()})")

    # collect only the frames we actually need across all clips
    needed_frames = set()
    for clip in clips:
        unit = clip.get('unit', 'sec')
        if unit == 'sec':
            s = max(0, math.floor(clip['start'] * fps))
            e = min(n_frames - 1, math.ceil(clip['stop'] * fps))
        else:
            s = max(0, int(clip['start']))
            e = min(n_frames - 1, int(clip['stop']))
        needed_frames.update(range(s, e + 1))

    for frame_idx, frame_df in trk[trk['frame'].isin(needed_frames)].groupby('frame'):
        fly_positions, fly_orientations = {}, {}
        for _, row in frame_df.drop_duplicates('id').iterrows():
            fly_id = int(row['id'])
            if pd.isna(row['pos_x']) or pd.isna(row['pos_y']):
                continue
            fly_positions[fly_id] = (int(row['pos_x']), int(row['pos_y']))
            if not pd.isna(row['ori']):
                fly_orientations[fly_id] = float(row['ori'])
        frame_lookup[frame_idx] = (fly_positions, fly_orientations)

    print(f"Built frame lookup for {len(frame_lookup)} frames")

In [36]:
ARROW_LEN = 20  # orientation arrow length in pixels

def _mark_frame(frame, frame_idx, frame_lookup, id_colors):
    # frame number in top-left corner
    cv2.putText(frame, f'frame {frame_idx}', (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

    if frame_idx not in frame_lookup:
        return
    fly_positions, fly_orientations = frame_lookup[frame_idx]
    for fly_id, (px, py) in fly_positions.items():
        color = id_colors.get(fly_id, (255, 255, 255))
        # filled dot
        cv2.circle(frame, (px, py), radius=6, color=color, thickness=-1)
        # orientation arrow
        if fly_id in fly_orientations:
            ori = fly_orientations[fly_id]
            ex = int(px + ARROW_LEN * np.cos(ori))
            ey = int(py + ARROW_LEN * np.sin(ori))
            cv2.arrowedLine(frame, (px, py), (ex, ey), color=color, thickness=2, tipLength=0.3)
        # id label — offset up-right from the dot
        cv2.putText(frame, f'id={fly_id}', (px + 10, py - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)


os.makedirs(save_dir, exist_ok=True)
cap = cv2.VideoCapture(avi_path)

for clip in clips:
    unit = clip.get('unit', 'sec')
    if unit == 'sec':
        start_frame = max(0, math.floor(clip['start'] * fps))
        stop_frame  = min(n_frames - 1, math.ceil(clip['stop'] * fps))
    else:
        start_frame = max(0, int(clip['start']))
        stop_frame  = min(n_frames - 1, int(clip['stop']))

    out_path = os.path.join(save_dir, f"{clip['name']}_frames{start_frame}-{stop_frame}.mp4")
    clip_len = stop_frame - start_frame + 1
    print(f"Writing {clip['name']}  frames {start_frame}–{stop_frame}  ({clip_len/fps:.1f} s) → {out_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    fourcc = cv2.VideoWriter_fourcc(*'avc1')
    out = cv2.VideoWriter(out_path, fourcc, fps, (frame_w, frame_h))

    for frame_idx in range(start_frame, stop_frame + 1):
        ret, frame = cap.read()
        if not ret:
            break
        if mark_flies:
            _mark_frame(frame, frame_idx, frame_lookup, id_colors)
        out.write(frame)

    out.release()

cap.release()
print("Done.")

Writing courting_over_deF  frames 76000–77000  (16.7 s) → /Users/juliechen/Library/CloudStorage/Dropbox-Dropbox@RU/Julie Chen/Ruta lab rotation 2026/fb_20mm/figures/20260511-1000_MMF-triad3_Dyak__4do_gh/clips/courting_over_deF_frames76000-77000.mp4
Writing guarding_circling  frames 98300–98800  (8.3 s) → /Users/juliechen/Library/CloudStorage/Dropbox-Dropbox@RU/Julie Chen/Ruta lab rotation 2026/fb_20mm/figures/20260511-1000_MMF-triad3_Dyak__4do_gh/clips/guarding_circling_frames98300-98800.mp4
Done.


## Sample clips by target velocity

Extracts short clips around frames where the courtship target is moving at different speeds,
organized into bins (e.g. 0–2, 2–4, 4–6 mm/s). Useful for deciding a pursuit velocity threshold.

Clips are saved to `rootdir/figures/target_vel_clip_examples/{assay_type}/{bin}/`.

In [3]:
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..', '..', '..'))

from analyses.triad.src import putil as tputil

# ── CONFIG ───────────────────────────────────────────────────────────────────
vel_rootdir = "/Volumes/Julie/fb_MMF_MFF_triad_38mm/"
vel_acq     = "20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh"

VEL_STEP        = 2.0   # mm/s bin width
N_CLIPS_PER_BIN = 3     # clips sampled per bin
CLIP_PRE_SEC    = 1.5   # seconds before the seed frame
CLIP_POST_SEC   = 1.5   # seconds after the seed frame
VEL_RANGE       = (0, 20)  # (min, max) in mm/s; set None to auto-range from data
# ─────────────────────────────────────────────────────────────────────────────

import glob, re
import pandas as pd

trk_path = os.path.join(vel_rootdir, 'processed_mats', f'{vel_acq}.parquet')
base_acq = re.sub(r'_ch\d+$', '', vel_acq)
avi_matches = glob.glob(os.path.join(vel_rootdir, 'raw_videos', base_acq, '*.avi'))
assert avi_matches, f"No .avi found for {vel_acq}"
vel_avi_path = avi_matches[0]

acq_df = pd.read_parquet(trk_path)
FPS    = int(acq_df['FPS'].iloc[0]) if 'FPS' in acq_df.columns else 60
vel_save_dir = os.path.join(vel_rootdir, 'figures', vel_acq, 'target_vel_clips')

tputil.sample_clips_by_metric(
    acq_df, vel_avi_path, vel_save_dir, metric='target_vel', fps=FPS, acq=vel_acq,
    action_col='courtship',
    step=VEL_STEP,
    n_clips=N_CLIPS_PER_BIN,
    clip_pre_sec=CLIP_PRE_SEC,
    clip_post_sec=CLIP_POST_SEC,
    metric_range=VEL_RANGE)

['/Volumes/Julie/fb_MMF_MFF_triad_38mm/figures/20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh/target_vel_clips/0.0-2.0/0.0-2.0_20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh_fr025349_act0_targ1.mp4',
 '/Volumes/Julie/fb_MMF_MFF_triad_38mm/figures/20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh/target_vel_clips/0.0-2.0/0.0-2.0_20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh_fr011113_act0_targ1.mp4',
 '/Volumes/Julie/fb_MMF_MFF_triad_38mm/figures/20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh/target_vel_clips/0.0-2.0/0.0-2.0_20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh_fr013024_act0_targ1.mp4',
 '/Volumes/Julie/fb_MMF_MFF_triad_38mm/figures/20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh/target_vel_clips/2.0-4.0/2.0-4.0_20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh_fr025323_act0_targ1.mp4',
 '/Volumes/Julie/fb_MMF_MFF_triad_38mm/figures/20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh/target_vel_clips/2.0-4.0/2.0-4.0_20260428-1147_MFF-triad5_Dmel_CantonS_5do_gh_fr002793_act0_targ1.mp4',
 '/Volumes/Julie/fb_